In [1]:
import numpy as np
import pandas as pd

In [2]:
def extract_temporal_features_per_rep(rep_df, fps=30):
    """
    Takes a dataframe containing exactly ONE repetition and calculates
    all 6 required Temporal & Phase Duration summary features.
    
    Parameters:
    - rep_df: Dataframe of a single repetition slice
    - fps: Framerate of the video to calculate absolute times (default=30)
    """
    # 1. Total duration metrics
    total_frames = len(rep_df)
    total_rep_duration = total_frames / fps

    # Find the frame representing the absolute bottom of the squat
    bottom_idx = rep_df['hip_y_smooth'].idxmax()
    
    # Split the dataframe into descent and ascent sets
    # .loc includes the boundaries, meaning bottom_idx is present in both 
    # to perfectly transition phases.
    descent_df = rep_df.loc[:bottom_idx]
    ascent_df = rep_df.loc[bottom_idx:]

    descent_frames = len(descent_df)
    ascent_frames = len(ascent_df)

    descent_duration_seconds = descent_frames / fps
    ascent_duration_seconds = ascent_frames / fps

    # --- 2. Calculate descent to ascent ratio ---
    if ascent_duration_seconds > 0:
        descent_to_ascent_ratio = descent_duration_seconds / ascent_duration_seconds
    else:
        descent_to_ascent_ratio = 0.0

    # --- 3. Calculate bottom pause duration ---
    # We define a "pause" as frames around the bottom where hip speed drops
    # below a tiny baseline threshold, meaning the lifter stopped moving vertically.
    # Adjust the threshold (0.015) based on your tracking coordinate scale.
    pause_threshold = 0.015 
    
    # Scan a small window of frames around the bottom peak
    bottom_window_start = max(rep_df.index[0], bottom_idx - 5)
    bottom_window_end = min(rep_df.index[-1], bottom_idx + 5)
    bottom_window_df = rep_df.loc[bottom_window_start:bottom_window_end]
    
    pause_frames = (bottom_window_df['hip_speed'] < pause_threshold).sum()
    bottom_pause_duration_seconds = pause_frames / fps

    # --- 4. Calculate time to max jerk ---
    # Jerk is the third derivative of position. Since you have velocity columns,
    # acceleration is diff() of velocity, and jerk is diff() of acceleration.
    # Let's compute it locally for the Y axis of the hip.
    hip_velocities = rep_df['hip_velocity_y'].values
    hip_accelerations = np.diff(hip_velocities, prepend=0)
    hip_jerks = np.abs(np.diff(hip_accelerations, prepend=0))
    
    if len(hip_jerks) > 0:
        max_jerk_local_idx = np.argmax(hip_jerks)
        # Progress through the rep from 0.0 (start) to 1.0 (end)
        time_to_max_jerk = max_jerk_local_idx / total_frames
    else:
        time_to_max_jerk = 0.0

    return {
        'descent_duration_seconds': descent_duration_seconds,
        'ascent_duration_seconds': ascent_duration_seconds,
        'bottom_pause_duration_seconds': bottom_pause_duration_seconds,
        'descent_to_ascent_ratio': descent_to_ascent_ratio,
        'total_rep_duration': total_rep_duration,
        'time_to_max_jerk': time_to_max_jerk
    }